# Prepare CC-Perturb-seq analysis files for IGVF

This notebook performs **read-only Portal auditing and local file staging only**. It does not POST, PATCH, or upload any object.

The AnalysisSet design is fixed as **four existing timepoint-specific AnalysisSets**:

| Day | Active AnalysisSet |
|---|---|
| D0 | `IGVFDS5257JGQO` |
| D1 | `IGVFDS3530HHEU` |
| D2 | `IGVFDS7380DUFB` |
| D3 | `IGVFDS6826YKCZ` |

Every new MatrixFile or TabularFile is assigned strictly by its `day`. The separately supplied AnalysisSets containing files already uploaded are read-only examples and are never used as a fifth active AnalysisSet.

The remaining unresolved decisions are:

1. whether the combined D0–D3 PerturbNMF h5mu is submitted once, omitted, or represented another way;
2. whether current local/global outputs qualify as cis/trans;
3. where the 16 existing differential-result source tables are located.

The PerturbNMF files already uploaded by the user are queried as examples and are not regenerated by default.


In [1]:
from __future__ import annotations

import json
import os
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display, Markdown

# Resolve the package root whether Jupyter starts in the package root or notebooks/.
CWD = Path.cwd().resolve()
if (CWD / "config" / "config.example.yaml").exists():
    PACKAGE_ROOT = CWD
elif (CWD.parent / "config" / "config.example.yaml").exists():
    PACKAGE_ROOT = CWD.parent
else:
    raise FileNotFoundError(
        "Run this notebook from the package root or its notebooks/ directory. "
        f"Current directory: {CWD}"
    )

sys.path.insert(0, str(PACKAGE_ROOT / "src"))

from igvf_analysis_prep import (
    DecisionRequired,
    PrepError,
    audit_analysis_sets,
    build_crosswalk,
    build_metadata_tsvs,
    convert_table_to_tsv_gz,
    derive_guide_quantifications,
    derive_per_cell_quality_report,
    dry_run_commands,
    infer_column_dictionary,
    inspect_h5ad,
    inspect_h5mu,
    load_tsv,
    read_yaml,
    require_h5mu_strategy,
    save_tsv,
    split_h5ad_by_day,
    validate_tabular_semantics,
    validate_timepoint_analysis_sets,
    validate_tsv_gz,
    write_json,
)

WORK_DIR = PACKAGE_ROOT / "work"
WORK_DIR.mkdir(parents=True, exist_ok=True)
print(f"PACKAGE_ROOT = {PACKAGE_ROOT}")
print(f"WORK_DIR     = {WORK_DIR}")

PACKAGE_ROOT = /oak/stanford/groups/engreitz/Users/tri/CodeBase/programDE/notebooks/cc_perturbseq_igvf_analysis_prep_4_timepoints
WORK_DIR     = /oak/stanford/groups/engreitz/Users/tri/CodeBase/programDE/notebooks/cc_perturbseq_igvf_analysis_prep_4_timepoints/work


## 1. Load the configuration and governing inventory

Create `config/config.yaml` from the example before changing any setting. If it is absent, this notebook uses the read-only example configuration.

In [2]:
CONFIG_PATH = PACKAGE_ROOT / "config" / "config.yaml"
if not CONFIG_PATH.exists():
    CONFIG_PATH = PACKAGE_ROOT / "config" / "config.example.yaml"
    print("Using config.example.yaml. Staging writes remain disabled by default.")

cfg = read_yaml(CONFIG_PATH)
expected_path = PACKAGE_ROOT / "config" / "expected_files.tsv"
overrides_path = PACKAGE_ROOT / cfg["source_paths"]["source_path_overrides"]
portal_examples_path = PACKAGE_ROOT / "config" / "portal_examples.tsv"
sequence_accessions_path = PACKAGE_ROOT / cfg["project"]["sequence_accession_manifest"]

active_mapping_name = cfg["analysis_set_mappings"].get("active_mapping")
if not isinstance(active_mapping_name, str) or not active_mapping_name:
    raise DecisionRequired("Set analysis_set_mappings.active_mapping to workbook_current")
active_sets = validate_timepoint_analysis_sets(
    cfg["analysis_set_mappings"].get(active_mapping_name, {})
)

expected = load_tsv(expected_path)
expected["active_file_set"] = expected["day"].map(active_sets)

day_counts = expected.groupby("day").size().to_dict()
required_counts = {day: 13 for day in ["D0", "D1", "D2", "D3"]}
if day_counts != required_counts:
    raise PrepError(
        "Expected exactly 13 planned files in each of D0-D3. "
        f"Observed: {day_counts}"
    )

file_set_mismatch = expected[
    expected["workbook_file_set"].ne(expected["active_file_set"])
]
if not file_set_mismatch.empty:
    raise PrepError(
        "Expected-file inventory contains file_set values inconsistent with "
        "the authoritative four-timepoint mapping."
    )

portal_examples = load_tsv(portal_examples_path)

print(f"Config: {CONFIG_PATH}")
print(f"Expected files: {len(expected)}")
print("Active timepoint AnalysisSets:")
print(json.dumps(active_sets, indent=2))
print(expected.groupby(["profile", "upload_action"]).size().to_string())

display(expected[[
    "index", "day", "profile", "expected_file_role", "active_file_set",
    "workbook_file_set", "provided_example_file_set",
    "source_status", "conversion_action", "upload_action"
]])


Using config.example.yaml. Staging writes remain disabled by default.
Config: /oak/stanford/groups/engreitz/Users/tri/CodeBase/programDE/notebooks/cc_perturbseq_igvf_analysis_prep_4_timepoints/config/config.example.yaml
Expected files: 52
Active timepoint AnalysisSets:
{
  "D0": "IGVFDS5257JGQO",
  "D1": "IGVFDS3530HHEU",
  "D2": "IGVFDS7380DUFB",
  "D3": "IGVFDS6826YKCZ"
}
profile       upload_action                         
matrix_file   BLOCK_UNTIL_DACC_DECISION                  4
              PREPARE_IF_VALID                           4
tabular_file  PREPARE_IF_VALID                          24
              SKIP_ALREADY_UPLOADED_RECONCILE_PORTAL    20


,index,day,profile,expected_file_role,active_file_set,workbook_file_set,provided_example_file_set,source_status,conversion_action,upload_action
0,1,D0,matrix_file,annotated cell by gene and guide matrix,IGVFDS5257JGQO,IGVFDS5257JGQO,,KNOWN_CANDIDATE_NOT_CHECKED_ON_SHERLOCK,SPLIT_COMBINED_H5AD_BY_DAY,PREPARE_IF_VALID
1,2,D0,matrix_file,cell by gene program matrix,IGVFDS5257JGQO,IGVFDS5257JGQO,,BLOCKED_COMBINED_H5MU_DECISION,DO_NOT_SPLIT_OR_COPY_PER_DAY,BLOCK_UNTIL_DACC_DECISION
2,3,D1,matrix_file,annotated cell by gene and guide matrix,IGVFDS3530HHEU,IGVFDS3530HHEU,,KNOWN_CANDIDATE_NOT_CHECKED_ON_SHERLOCK,SPLIT_COMBINED_H5AD_BY_DAY,PREPARE_IF_VALID
3,4,D1,matrix_file,cell by gene program matrix,IGVFDS3530HHEU,IGVFDS3530HHEU,,BLOCKED_COMBINED_H5MU_DECISION,DO_NOT_SPLIT_OR_COPY_PER_DAY,BLOCK_UNTIL_DACC_DECISION
4,5,D2,matrix_file,annotated cell by gene and guide matrix,IGVFDS7380DUFB,IGVFDS7380DUFB,,KNOWN_CANDIDATE_NOT_CHECKED_ON_SHERLOCK,SPLIT_COMBINED_H5AD_BY_DAY,PREPARE_IF_VALID
5,6,D2,matrix_file,cell by gene program matrix,IGVFDS7380DUFB,IGVFDS7380DUFB,,BLOCKED_COMBINED_H5MU_DECISION,DO_NOT_SPLIT_OR_COPY_PER_DAY,BLOCK_UNTIL_DACC_DECISION
6,7,D3,matrix_file,annotated cell by gene and guide matrix,IGVFDS6826YKCZ,IGVFDS6826YKCZ,,KNOWN_CANDIDATE_NOT_CHECKED_ON_SHERLOCK,SPLIT_COMBINED_H5AD_BY_DAY,PREPARE_IF_VALID
7,8,D3,matrix_file,cell by gene program matrix,IGVFDS6826YKCZ,IGVFDS6826YKCZ,,BLOCKED_COMBINED_H5MU_DECISION,DO_NOT_SPLIT_OR_COPY_PER_DAY,BLOCK_UNTIL_DACC_DECISION
8,9,D0,tabular_file,guide quantifications,IGVFDS5257JGQO,IGVFDS5257JGQO,,DERIVABLE_AFTER_H5AD_SPLIT,DERIVE_FROM_DAY_H5AD,PREPARE_IF_VALID
9,10,D0,tabular_file,per-cell quality report,IGVFDS5257JGQO,IGVFDS5257JGQO,,DERIVABLE_AFTER_H5AD_SPLIT,DERIVE_FROM_DAY_H5AD,PREPARE_IF_VALID


## 2. Decision audit

The notebook reports conflicts instead of choosing silently.

In [3]:
decision_rows = [
    {
        "decision": "AnalysisSet design",
        "status": "SELECTED_FOUR_TIMEPOINT_SETS",
        "detail": json.dumps(active_sets),
    },
    {
        "decision": "Timepoint isolation",
        "status": "ENFORCED",
        "detail": "Each new file_set is mapped from day only; D0-D3 use four distinct AnalysisSets and no combined fifth set is active.",
    },
    {
        "decision": "PerturbNMF h5mu representation",
        "status": cfg["options"].get("perturbnmf_matrix_strategy", "BLOCK_UNTIL_DACC_DECISION"),
        "detail": cfg["source_paths"].get("combined_h5mu", ""),
    },
    {
        "decision": "Local/global content types",
        "status": "PENDING_DACC_CONFIRMATION",
        "detail": "Do not assume local=cis or global=trans without the actual analysis definitions.",
    },
    {
        "decision": "Already uploaded PerturbNMF tables",
        "status": "SKIP_NEW_UPLOAD",
        "detail": "Use supplied Portal sets only as read-only examples; do not assign new files to them.",
    },
    {
        "decision": "AnalysisStep count",
        "status": "FOLLOW_WORKBOOK_3_STEPS",
        "detail": "The detailed plan describes four conceptual steps, but the current workbook/prompt intentionally folds reporting into Step 3.",
    },
]
decisions = pd.DataFrame(decision_rows)
display(decisions)
save_tsv(decisions, WORK_DIR / "decision_audit.tsv")


,decision,status,detail
0,AnalysisSet design,SELECTED_FOUR_TIMEPOINT_SETS,"{""D0"": ""IGVFDS5257JGQO"", ""D1"": ""IGVFDS3530HHEU..."
1,Timepoint isolation,ENFORCED,Each new file_set is mapped from day only; D0-...
2,PerturbNMF h5mu representation,BLOCK_UNTIL_DACC_DECISION,/oak/stanford/groups/engreitz/Users/ymo/IGVF_c...
3,Local/global content types,PENDING_DACC_CONFIRMATION,Do not assume local=cis or global=trans withou...
4,Already uploaded PerturbNMF tables,SKIP_NEW_UPLOAD,Use supplied Portal sets only as read-only exa...
5,AnalysisStep count,FOLLOW_WORKBOOK_3_STEPS,The detailed plan describes four conceptual st...


PosixPath('/oak/stanford/groups/engreitz/Users/tri/CodeBase/programDE/notebooks/cc_perturbseq_igvf_analysis_prep_4_timepoints/work/decision_audit.tsv')

## 3. Check known Sherlock paths

These paths were extracted from the two attached analysis notebooks. This cell checks them on Sherlock instead of assuming they still exist.

In [4]:
candidate_paths = load_tsv(PACKAGE_ROOT / "config" / "sherlock_candidate_paths.tsv")
candidate_paths["exists"] = candidate_paths["path"].map(
    lambda value: Path(value).is_file() if value and not value.startswith("TBD_") else False
)
candidate_paths["size_bytes"] = candidate_paths["path"].map(
    lambda value: Path(value).stat().st_size if value and Path(value).is_file() else ""
)
display(candidate_paths)
save_tsv(candidate_paths, WORK_DIR / "sherlock_path_audit.tsv")

,role,path,source,confidence,action,exists,size_bytes
0,combined annotated h5ad,/oak/stanford/groups/engreitz/Users/opushkar/c...,fig3 and jesse_umap notebooks,high,inspect on Sherlock; split by day only after d...,True,4218513308
1,combined PerturbNMF h5mu,/oak/stanford/groups/engreitz/Users/ymo/IGVF_c...,jesse_umap notebook,high,inspect; do not create four copies,True,1491701396
2,cNMF usage matrix,/oak/stanford/groups/engreitz/Users/ymo/IGVF_c...,fig3 notebook,high,supporting input/example; not automatically a ...,True,51382087
3,program metadata/annotations,/oak/stanford/groups/engreitz/Users/tri/CodeBa...,jesse_umap notebook repository convention,medium,supporting metadata; confirm actual path exists,True,24047
4,guide metadata v43,TBD_ABSOLUTE_PATH/300genes_guide_metadata_v43.tsv,prompt says already uploaded,missing,provide absolute Sherlock path if needed for c...,False,


PosixPath('/oak/stanford/groups/engreitz/Users/tri/CodeBase/programDE/notebooks/cc_perturbseq_igvf_analysis_prep_4_timepoints/work/sherlock_path_audit.tsv')

## 4. Audit active and example AnalysisSets (read-only)

This cell retrieves:

- the four **active** D0–D3 AnalysisSets from the metadata workbook, so their existence and `input_file_sets` can be checked;
- the five separately supplied AnalysisSets containing files already uploaded, so their schemas and warnings can be reviewed.

The supplied prior-upload sets are audit examples only. They are never used as `file_set` for new files. Unreleased records may require `IGVF_API_KEY` and `IGVF_SECRET_KEY` exported before Jupyter starts.

`ERROR` and `NOT_COMPLIANT` items are blockers. `WARNING` items are preserved for review; they are not automatically dismissed.


In [5]:
RUN_PORTAL_AUDIT = bool(cfg["options"].get("run_portal_audit", True))
portal_set_df = pd.DataFrame()
portal_file_df = pd.DataFrame()
portal_audit_df = pd.DataFrame()

if RUN_PORTAL_AUDIT:
    accessions = portal_examples["analysis_set_accession"].tolist()
    try:
        portal_set_df, portal_file_df, portal_audit_df = audit_analysis_sets(accessions)
        save_tsv(portal_set_df, WORK_DIR / "portal_analysis_sets.tsv")
        save_tsv(portal_file_df, WORK_DIR / "portal_files.tsv")
        save_tsv(portal_audit_df, WORK_DIR / "portal_audits.tsv")
        print(f"AnalysisSets found: {len(portal_set_df)}")
        print(f"Linked files found: {len(portal_file_df)}")
        print(f"Audit messages:     {len(portal_audit_df)}")
        display(portal_set_df)
        display(portal_file_df[[c for c in [
            "analysis_set", "accession", "object_type", "content_type", "file_format",
            "status", "upload_status", "checkfiles_version", "audit_counts", "portal_url"
        ] if c in portal_file_df.columns]])
        if not portal_audit_df.empty:
            severity_order = {"ERROR": 0, "NOT_COMPLIANT": 1, "WARNING": 2, "INTERNAL_ACTION": 3}
            portal_audit_df["_order"] = portal_audit_df["audit_level"].map(severity_order).fillna(99)
            display(portal_audit_df.sort_values(["_order", "accession"]).drop(columns="_order"))
            blockers = portal_audit_df[portal_audit_df["audit_level"].isin(["ERROR", "NOT_COMPLIANT"])]
            print(f"Blocker-level audit messages: {len(blockers)}")
        else:
            print("No audit messages were returned for the queried objects.")
    except Exception as exc:
        print("Portal audit could not be completed in this session.")
        print(f"{type(exc).__name__}: {exc}")
        print("Export valid IGVF_API_KEY and IGVF_SECRET_KEY in the shell, restart the kernel, and rerun this cell.")
else:
    print("Portal audit disabled in config.")

Portal audit could not be completed in this session.
PrepError: Portal GET failed for https://api.data.igvf.org/IGVFDS5257JGQO/: 403 Client Error: Forbidden for url: https://api.data.igvf.org/IGVFDS5257JGQO/?format=json
Export valid IGVF_API_KEY and IGVF_SECRET_KEY in the shell, restart the kernel, and rerun this cell.


### Reconcile already-uploaded file categories

This matches the workbook rows marked “already uploaded” to files returned by the supplied example AnalysisSets. One exact content-type match is expected; zero or multiple matches require manual review.

In [6]:
existing_reconciliation = pd.DataFrame()
if not portal_file_df.empty:
    skip_rows = expected[expected["upload_action"].eq("SKIP_ALREADY_UPLOADED_RECONCILE_PORTAL")].copy()
    records = []
    for _, row in skip_rows.iterrows():
        matches = portal_file_df[
            portal_file_df["analysis_set"].eq(row["provided_example_file_set"]) &
            portal_file_df["content_type"].eq(row["content_type"])
        ]
        if len(matches) == 1:
            match = matches.iloc[0]
            status = "FOUND_EXACT"
            accession = match.get("accession", "")
            url = match.get("portal_url", "")
            warning = match.get("audit_counts", "")
        elif len(matches) == 0:
            status = "NOT_FOUND"
            accession = ""
            url = ""
            warning = ""
        else:
            status = "AMBIGUOUS_MULTIPLE"
            accession = "|".join(matches["accession"].astype(str))
            url = "|".join(matches["portal_url"].astype(str))
            warning = "|".join(matches["audit_counts"].astype(str))
        records.append({
            "day": row["day"],
            "expected_file_role": row["expected_file_role"],
            "expected_alias": row["alias"],
            "portal_example_analysis_set": row["provided_example_file_set"],
            "existing_file_accession": accession,
            "existing_portal_url": url,
            "audit_counts": warning,
            "reconciliation_status": status,
        })
    existing_reconciliation = pd.DataFrame(records)
    save_tsv(existing_reconciliation, WORK_DIR / "existing_uploaded_reconciliation.tsv")
    display(existing_reconciliation)
    print(existing_reconciliation["reconciliation_status"].value_counts().to_string())
else:
    print("Portal files are unavailable; reconciliation template remains under config/.")

Portal files are unavailable; reconciliation template remains under config/.


## 5. Inspect the actual h5ad and h5mu structures

The combined h5ad can be split by a validated day field. The combined h5mu is inspected but not split automatically.

In [7]:
h5ad_path = Path(cfg["source_paths"].get("combined_h5ad", ""))
h5mu_path = Path(cfg["source_paths"].get("combined_h5mu", ""))
day_candidates = cfg["options"].get("day_column_candidates", ["day", "timepoint", "Day"])

h5ad_report = inspect_h5ad(h5ad_path, day_column_candidates=day_candidates)
write_json(h5ad_report, WORK_DIR / "h5ad_inspection.json")
display(pd.DataFrame([{
    k: json.dumps(v) if isinstance(v, (list, dict)) else v
    for k, v in h5ad_report.items()
}]))

h5mu_report = inspect_h5mu(h5mu_path, day_column_candidates=day_candidates)
write_json(h5mu_report, WORK_DIR / "h5mu_inspection.json")
display(pd.DataFrame([{
    k: json.dumps(v) if isinstance(v, (list, dict)) else v
    for k, v in h5mu_report.items()
}]))

try:
    require_h5mu_strategy(h5mu_report, cfg["options"].get("perturbnmf_matrix_strategy", ""))
except DecisionRequired as exc:
    print("DECISION REQUIRED:", exc)

,path,exists,status,n_obs,n_vars,obs_names_unique,var_names_unique,obs_columns,var_columns,obsm_keys,...,uns_keys,size_bytes,md5sum,guide_information_keys,guide_information_present,rna_matrix_present,day_column,day_values_raw,day_values_normalized,unmapped_day_count
0,/oak/stanford/groups/engreitz/Users/opushkar/c...,True,PASS,121616,17538,True,True,"[""guides_per_cell"", ""guide_umi_counts"", ""pct_c...","[""ensembl_id"", ""gene_symbol""]","[""X_pca"", ""X_umap"", ""guide_assignment""]",...,"[""guide_names"", ""leiden_res_0.0"", ""leiden_res_...",4218513308,fd62f2de1d690e4c564af72e83fb3360,"[""guide_assignment"", ""guide_names"", ""guide_umi...",True,True,day,"[""d0"", ""d1"", ""d2"", ""d3""]","[""D0"", ""D1"", ""D2"", ""D3""]",0


/oak/stanford/groups/engreitz/Users/tri/envs/sc-dl/lib/python3.11/site-packages/mudata/_core/mudata.py:1598: FutureWarning: From 0.4 .update() will not pull obs/var columns from individual modalities by default anymore. Set mudata.set_options(pull_on_update=False) to adopt the new behaviour, which will become the default. Use new pull_obs/pull_var and push_obs/push_var methods for more flexibility.
  self._update_attr("var", axis=0, join_common=join_common)
/oak/stanford/groups/engreitz/Users/tri/envs/sc-dl/lib/python3.11/site-packages/mudata/_core/mudata.py:1461: FutureWarning: From 0.4 .update() will not pull obs/var columns from individual modalities by default anymore. Set mudata.set_options(pull_on_update=False) to adopt the new behaviour, which will become the default. Use new pull_obs/pull_var and push_obs/push_var methods for more flexibility.
  self._update_attr("obs", axis=1, join_common=join_common)


,path,exists,status,modalities,day_locations,distinct_days,combined_across_days,size_bytes,md5sum
0,/oak/stanford/groups/engreitz/Users/ymo/IGVF_c...,True,PASS,"{""rna"": {""n_obs"": 121616, ""n_vars"": 17538, ""ob...","[{""location"": ""mod[rna].obs"", ""column"": ""day"",...","[""D0"", ""D1"", ""D2"", ""D3""]",True,1491701396,833be2954a5c4dee5b2072791a3e8664


DECISION REQUIRED: The actual PerturbNMF h5mu appears to contain multiple timepoints. The workbook assumes one h5mu per day. Do not create four copies. Choose a DACC-approved combined MatrixFile strategy first.


## 6. Build the source-to-submission crosswalk

Fill `config/source_path_overrides.tsv` with the existing differential-output paths. The crosswalk retains every planned row, including missing and already-uploaded objects.

In [8]:
crosswalk = build_crosswalk(expected_path, overrides_path)

# Enforce the four timepoint-specific AnalysisSets in the source-to-submission crosswalk.
crosswalk["file_set"] = crosswalk["day"].map(active_sets)
if crosswalk["file_set"].isna().any():
    bad_days = sorted(crosswalk.loc[crosswalk["file_set"].isna(), "day"].unique())
    raise DecisionRequired(f"Rows have no active timepoint AnalysisSet: {bad_days}")

# Planned staging outputs are the workbook submitted_file_name paths.
crosswalk["submitted_file_name"] = crosswalk["planned_submission_path"]
crosswalk["requires_conversion"] = crosswalk["conversion_action"].ne("NONE")
crosswalk["requires_renaming"] = crosswalk.apply(
    lambda r: bool(r.get("source_path")) and Path(r["source_path"]).name != Path(r["planned_submission_path"]).name,
    axis=1,
)
crosswalk["requires_compression"] = crosswalk.apply(
    lambda r: r["file_format"] == "tsv" and not str(r.get("source_path", "")).endswith(".tsv.gz"),
    axis=1,
)

save_tsv(crosswalk, WORK_DIR / "source_to_submission_crosswalk.tsv")
display(crosswalk[[
    "index", "day", "file_set", "expected_file_role", "source_path", "exists",
    "planned_submission_path", "conversion_action", "upload_action", "notes"
]])

print("\nInventory summary")
print(crosswalk.groupby(["day", "file_set", "upload_action", "exists"]).size().to_string())


,index,day,file_set,expected_file_role,source_path,exists,planned_submission_path,conversion_action,upload_action,notes
0,1,D0,IGVFDS5257JGQO,annotated cell by gene and guide matrix,/oak/stanford/groups/engreitz/Users/opushkar/c...,True,/oak/stanford/groups/engreitz/Projects/IGVF/CC...,SPLIT_COMBINED_H5AD_BY_DAY,PREPARE_IF_VALID,Do not modify source; create day-specific subs...
1,2,D0,IGVFDS5257JGQO,cell by gene program matrix,/oak/stanford/groups/engreitz/Users/ymo/IGVF_c...,True,/oak/stanford/groups/engreitz/Projects/IGVF/CC...,DO_NOT_SPLIT_OR_COPY_PER_DAY,BLOCK_UNTIL_DACC_DECISION,Known h5mu is combined D0-D3; workbook assumes...
2,3,D1,IGVFDS3530HHEU,annotated cell by gene and guide matrix,/oak/stanford/groups/engreitz/Users/opushkar/c...,True,/oak/stanford/groups/engreitz/Projects/IGVF/CC...,SPLIT_COMBINED_H5AD_BY_DAY,PREPARE_IF_VALID,Do not modify source; create day-specific subs...
3,4,D1,IGVFDS3530HHEU,cell by gene program matrix,/oak/stanford/groups/engreitz/Users/ymo/IGVF_c...,True,/oak/stanford/groups/engreitz/Projects/IGVF/CC...,DO_NOT_SPLIT_OR_COPY_PER_DAY,BLOCK_UNTIL_DACC_DECISION,Known h5mu is combined D0-D3; workbook assumes...
4,5,D2,IGVFDS7380DUFB,annotated cell by gene and guide matrix,/oak/stanford/groups/engreitz/Users/opushkar/c...,True,/oak/stanford/groups/engreitz/Projects/IGVF/CC...,SPLIT_COMBINED_H5AD_BY_DAY,PREPARE_IF_VALID,Do not modify source; create day-specific subs...
5,6,D2,IGVFDS7380DUFB,cell by gene program matrix,/oak/stanford/groups/engreitz/Users/ymo/IGVF_c...,True,/oak/stanford/groups/engreitz/Projects/IGVF/CC...,DO_NOT_SPLIT_OR_COPY_PER_DAY,BLOCK_UNTIL_DACC_DECISION,Known h5mu is combined D0-D3; workbook assumes...
6,7,D3,IGVFDS6826YKCZ,annotated cell by gene and guide matrix,/oak/stanford/groups/engreitz/Users/opushkar/c...,True,/oak/stanford/groups/engreitz/Projects/IGVF/CC...,SPLIT_COMBINED_H5AD_BY_DAY,PREPARE_IF_VALID,Do not modify source; create day-specific subs...
7,8,D3,IGVFDS6826YKCZ,cell by gene program matrix,/oak/stanford/groups/engreitz/Users/ymo/IGVF_c...,True,/oak/stanford/groups/engreitz/Projects/IGVF/CC...,DO_NOT_SPLIT_OR_COPY_PER_DAY,BLOCK_UNTIL_DACC_DECISION,Known h5mu is combined D0-D3; workbook assumes...
8,9,D0,IGVFDS5257JGQO,guide quantifications,/oak/stanford/groups/engreitz/Users/opushkar/c...,True,/oak/stanford/groups/engreitz/Projects/IGVF/CC...,DERIVE_FROM_DAY_H5AD,PREPARE_IF_VALID,Candidate schema must be reviewed before porta...
9,10,D0,IGVFDS5257JGQO,per-cell quality report,/oak/stanford/groups/engreitz/Users/opushkar/c...,True,/oak/stanford/groups/engreitz/Projects/IGVF/CC...,DERIVE_FROM_DAY_H5AD,PREPARE_IF_VALID,Candidate schema must be reviewed before porta...



Inventory summary
day  file_set        upload_action                           exists
D0   IGVFDS5257JGQO  BLOCK_UNTIL_DACC_DECISION               True      1
                     PREPARE_IF_VALID                        False     4
                                                             True      3
                     SKIP_ALREADY_UPLOADED_RECONCILE_PORTAL  False     3
                                                             True      2
D1   IGVFDS3530HHEU  BLOCK_UNTIL_DACC_DECISION               True      1
                     PREPARE_IF_VALID                        False     4
                                                             True      3
                     SKIP_ALREADY_UPLOADED_RECONCILE_PORTAL  False     3
                                                             True      2
D2   IGVFDS7380DUFB  BLOCK_UNTIL_DACC_DECISION               True      1
                     PREPARE_IF_VALID                        False     4
                                     

## 7. Local staging

The default is `write_staged_files: false`. When enabled, this section:

- creates D0-D3 h5ad subsets from the combined h5ad;
- derives candidate guide-quantification and per-cell QC tables;
- converts the 16 supplied differential tables to deterministic `.tsv.gz` files;
- skips the five PerturbNMF table types already uploaded;
- refuses to create four copies of the combined h5mu.

In [9]:
WRITE_STAGING = bool(cfg["options"].get("write_staged_files", False))
OVERWRITE = bool(cfg["options"].get("overwrite_staged_files", False))
staging_records = []

if not WRITE_STAGING:
    print("Staging writes are disabled. Set write_staged_files: true only after reviewing the audit and decisions.")
else:
    # A. Split the combined h5ad.
    h5ad_rows = expected[(expected["profile"].eq("matrix_file")) & (expected["file_format"].eq("h5ad"))]
    h5ad_outputs = dict(zip(h5ad_rows["day"], h5ad_rows["planned_submission_path"]))
    split_report = split_h5ad_by_day(
        cfg["source_paths"]["combined_h5ad"],
        h5ad_outputs,
        day_column_candidates=day_candidates,
        overwrite=OVERWRITE,
    )
    split_report["role"] = "annotated cell by gene and guide matrix"
    staging_records.extend(split_report.to_dict("records"))
    display(split_report)

    # B. Derive guide/QC tables from each day-specific h5ad.
    for day in ["D0", "D1", "D2", "D3"]:
        day_h5ad = h5ad_outputs[day]
        guide_row = expected[(expected["day"].eq(day)) & (expected["expected_file_role"].eq("guide quantifications"))].iloc[0]
        qc_row = expected[(expected["day"].eq(day)) & (expected["expected_file_role"].eq("per-cell quality report"))].iloc[0]
        guide_out, _ = derive_guide_quantifications(day_h5ad, guide_row["planned_submission_path"])
        qc_out, _ = derive_per_cell_quality_report(day_h5ad, qc_row["planned_submission_path"])
        staging_records.extend([
            {"day": day, "role": "guide quantifications", "output_path": str(guide_out)},
            {"day": day, "role": "per-cell quality report", "output_path": str(qc_out)},
        ])

    # C. Convert existing differential tables supplied through the overrides TSV.
    override_df = load_tsv(overrides_path)
    for _, row in override_df.iterrows():
        source = str(row.get("source_path_override", "")).strip()
        if not source:
            continue
        target = expected[
            expected["day"].eq(row["day"]) &
            expected["expected_file_role"].eq(row["expected_file_role"])
        ]
        if len(target) != 1:
            raise PrepError(f"Expected exactly one target row for {row['day']} / {row['expected_file_role']}")
        target_path = Path(target.iloc[0]["planned_submission_path"])
        if target_path.exists() and not OVERWRITE:
            raise FileExistsError(f"Refusing to overwrite {target_path}")
        out, mapping = convert_table_to_tsv_gz(
            source,
            target_path,
            sheet_name=row.get("source_sheet", 0) or 0,
            rename_columns=bool(cfg["options"].get("rename_tabular_columns", False)),
        )
        mapping_path = WORK_DIR / "column_name_mappings" / f"{row['day']}_{target_path.stem}.tsv"
        save_tsv(mapping, mapping_path)
        staging_records.append({
            "day": row["day"],
            "role": row["expected_file_role"],
            "source_path": source,
            "output_path": str(out),
        })

    staging_df = pd.DataFrame(staging_records)
    save_tsv(staging_df, WORK_DIR / "staging_report.tsv")
    display(staging_df)

Staging writes are disabled. Set write_staged_files: true only after reviewing the audit and decisions.


## 8. Validate every planned output

This produces one status per workbook row. Only `READY` rows can enter draft metadata TSVs.

In [10]:
validation_rows = []
for _, row in expected.iterrows():
    path = Path(row["planned_submission_path"])
    status = "MISSING"
    detail = ""
    size = ""
    checksum = ""

    if row["upload_action"] == "SKIP_ALREADY_UPLOADED_RECONCILE_PORTAL":
        status = "ALREADY_UPLOADED_SKIP"
        detail = "Use the read-only portal audit; do not create a duplicate."
    elif row["upload_action"] == "BLOCK_UNTIL_DACC_DECISION":
        status = "BLOCKED_DECISION"
        detail = "Combined h5mu conflicts with four day-specific workbook rows."
    elif path.is_file():
        if row["file_format"] == "h5ad":
            report = inspect_h5ad(path, day_column_candidates=day_candidates)
            problems = []
            if report.get("status") != "PASS":
                problems.append(report.get("error", "h5ad open/read failure"))
            if not report.get("obs_names_unique", False):
                problems.append("cell IDs are not unique")
            if not report.get("var_names_unique", False):
                problems.append("gene IDs/names are not unique")
            if not report.get("rna_matrix_present", False):
                problems.append("RNA matrix is absent or empty")
            if not report.get("guide_information_present", False):
                problems.append("no guide-related obs/obsm/uns keys detected")
            observed_days = set(report.get("day_values_normalized", []))
            if observed_days != {row["day"]}:
                problems.append(f"expected only {row['day']}; observed normalized days={sorted(observed_days)}")
            if report.get("unmapped_day_count", 0):
                problems.append(f"{report['unmapped_day_count']} cells have unmapped day labels")
            status = "READY" if not problems else "INVALID"
            detail = "; ".join(filter(None, problems))
            size = report.get("size_bytes", "")
            checksum = report.get("md5sum", "")
        elif row["file_format"] == "h5mu":
            report = inspect_h5mu(path, day_column_candidates=day_candidates)
            status = "READY" if report.get("status") == "PASS" and not report.get("combined_across_days") else "BLOCKED_DECISION"
            detail = report.get("error", "") or ("combined D0-D3 h5mu" if report.get("combined_across_days") else "")
            size = report.get("size_bytes", "")
            checksum = report.get("md5sum", "")
        elif row["file_format"] == "tsv":
            report = validate_tsv_gz(path)
            semantic = validate_tabular_semantics(path) if report.get("status") == "PASS" else {"status": "SKIPPED"}
            problems = []
            if report.get("status") != "PASS":
                problems.append(report.get("error", "structural validation failed"))
            if semantic.get("status") == "ERROR":
                problems.append(
                    "numeric semantic checks failed: " + json.dumps({
                        "range_violations": semantic.get("range_violations", []),
                        "nonnumeric_values": semantic.get("nonnumeric_values", []),
                        "nonfinite_values": semantic.get("nonfinite_values", []),
                    })
                )
            status = "READY" if not problems else "INVALID"
            detail = "; ".join(filter(None, problems))
            size = report.get("size_bytes", "")
            checksum = report.get("md5sum", "")
        else:
            status = "INVALID"
            detail = f"Unsupported file format: {row['file_format']}"

    validation_rows.append({
        "index": row["index"],
        "alias": row["alias"],
        "day": row["day"],
        "expected_file_role": row["expected_file_role"],
        "submitted_file_name": str(path),
        "validation_status": status,
        "validation_detail": detail,
        "size_bytes": size,
        "md5sum": checksum,
    })

validation = pd.DataFrame(validation_rows)
save_tsv(validation, WORK_DIR / "validation_report.tsv")
display(validation)
print(validation["validation_status"].value_counts(dropna=False).to_string())

,index,alias,day,expected_file_role,submitted_file_name,validation_status,validation_detail,size_bytes,md5sum
0,1,jesse-engreitz:matrix_file_WTC11-EC_300gene_CC...,D0,annotated cell by gene and guide matrix,/oak/stanford/groups/engreitz/Projects/IGVF/CC...,MISSING,,,
1,2,jesse-engreitz:matrix_file_WTC11-EC_300gene_Pe...,D0,cell by gene program matrix,/oak/stanford/groups/engreitz/Projects/IGVF/CC...,BLOCKED_DECISION,Combined h5mu conflicts with four day-specific...,,
2,3,jesse-engreitz:matrix_file_WTC11-EC_300gene_CC...,D1,annotated cell by gene and guide matrix,/oak/stanford/groups/engreitz/Projects/IGVF/CC...,MISSING,,,
3,4,jesse-engreitz:matrix_file_WTC11-EC_300gene_Pe...,D1,cell by gene program matrix,/oak/stanford/groups/engreitz/Projects/IGVF/CC...,BLOCKED_DECISION,Combined h5mu conflicts with four day-specific...,,
4,5,jesse-engreitz:matrix_file_WTC11-EC_300gene_CC...,D2,annotated cell by gene and guide matrix,/oak/stanford/groups/engreitz/Projects/IGVF/CC...,MISSING,,,
5,6,jesse-engreitz:matrix_file_WTC11-EC_300gene_Pe...,D2,cell by gene program matrix,/oak/stanford/groups/engreitz/Projects/IGVF/CC...,BLOCKED_DECISION,Combined h5mu conflicts with four day-specific...,,
6,7,jesse-engreitz:matrix_file_WTC11-EC_300gene_CC...,D3,annotated cell by gene and guide matrix,/oak/stanford/groups/engreitz/Projects/IGVF/CC...,MISSING,,,
7,8,jesse-engreitz:matrix_file_WTC11-EC_300gene_Pe...,D3,cell by gene program matrix,/oak/stanford/groups/engreitz/Projects/IGVF/CC...,BLOCKED_DECISION,Combined h5mu conflicts with four day-specific...,,
8,9,jesse-engreitz:tabular_file_WTC11-EC_300gene_D...,D0,guide quantifications,/oak/stanford/groups/engreitz/Projects/IGVF/CC...,MISSING,,,
9,10,jesse-engreitz:tabular_file_WTC11-EC_300gene_D...,D0,per-cell quality report,/oak/stanford/groups/engreitz/Projects/IGVF/CC...,MISSING,,,


validation_status
MISSING                  28
ALREADY_UPLOADED_SKIP    20
BLOCKED_DECISION          4


## 9. Draft the column dictionary for staged TSVs

Descriptions, requiredness, and units remain `TBD` unless supported by the actual source analysis. The notebook does not invent them.

In [11]:
dictionary_parts = []
ready_tsv = expected.merge(validation[["alias", "validation_status"]], on="alias", how="left")
ready_tsv = ready_tsv[(ready_tsv["file_format"].eq("tsv")) & (ready_tsv["validation_status"].eq("READY"))]
for _, row in ready_tsv.iterrows():
    part = infer_column_dictionary(
        row["planned_submission_path"],
        table_role=row["expected_file_role"],
        source_analysis_step=row["analysis_step_version"],
    )
    part.insert(0, "day", row["day"])
    part.insert(0, "alias", row["alias"])
    dictionary_parts.append(part)

column_dictionary = pd.concat(dictionary_parts, ignore_index=True) if dictionary_parts else pd.DataFrame()
if not column_dictionary.empty:
    save_tsv(column_dictionary, WORK_DIR / "column_dictionary.tsv")
    display(column_dictionary)
else:
    print("No READY TSV files yet; no column dictionary generated.")

No READY TSV files yet; no column dictionary generated.


## 10. Build draft metadata TSVs and dry-run commands

The four timepoint AnalysisSets are already selected in the configuration. Metadata generation still requires each candidate file to validate as `READY`. This section writes metadata TSVs only; it does not call `iu_register.py`.


In [12]:
metadata_paths = {}
try:
    # Revalidate here so a local config edit cannot silently add a combined or fifth set.
    active_sets = validate_timepoint_analysis_sets(active_sets)
    metadata_paths = build_metadata_tsvs(
        expected_path,
        WORK_DIR / "validation_report.tsv",
        WORK_DIR / "metadata",
        active_analysis_sets=active_sets,
        sequence_accession_manifest=sequence_accessions_path,
    )
    print("Draft metadata files:")
    for key, value in metadata_paths.items():
        print(f"  {key}: {value}")
except (DecisionRequired, PrepError) as exc:
    print("Metadata finalization blocked:", exc)

commands = dry_run_commands(
    metadata_paths,
    iu_register=cfg["tools"]["iu_register"],
    python_executable=cfg["tools"]["igvf_python"],
) if metadata_paths else []

command_file = WORK_DIR / "dry_run_commands.sh"
command_file.write_text(
    "#!/bin/bash\nset -euo pipefail\n\n" + "\n\n".join(commands) + "\n",
    encoding="utf-8",
)
command_file.chmod(0o755)
print("\nProposed dry-run commands (not executed):")
print("\n".join(commands) if commands else "None yet.")


Draft metadata files:
  eligible_upload_manifest: /oak/stanford/groups/engreitz/Users/tri/CodeBase/programDE/notebooks/cc_perturbseq_igvf_analysis_prep_4_timepoints/work/metadata/eligible_upload_manifest.tsv

Proposed dry-run commands (not executed):
None yet.


## 11. Final first-pass summary

Review the generated tables under `work/`, then answer the questions in `docs/questions_needed.md`. Do not interpret a clean local validation or dry run as proof that live upload will succeed.

In [ ]:
summary = {
    "expected_rows": int(len(expected)),
    "validation_counts": validation["validation_status"].value_counts(dropna=False).to_dict(),
    "portal_analysis_sets_found": int(len(portal_set_df)),
    "portal_files_found": int(len(portal_file_df)),
    "portal_audit_messages": int(len(portal_audit_df)),
    "active_analysis_set_mapping": active_mapping_name,
    "active_analysis_sets": active_sets,
    "write_staged_files": WRITE_STAGING,
    "metadata_tsvs_created": {k: str(v) for k, v in metadata_paths.items()},
}
write_json(summary, WORK_DIR / "run_summary.json")
print(json.dumps(summary, indent=2))
print("\nNext: resolve the remaining h5mu, differential-path, and DACC content-type questions before enabling staging writes.")

### Generate summary

In [1]:
from __future__ import annotations

import json
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd

# -----------------------------------------------------------------------------
# Resolve package paths. This works when the notebook is opened from either
# the package root or package_root/notebooks/.
# -----------------------------------------------------------------------------
CWD = Path.cwd().resolve()
if (CWD / "config" / "expected_files.tsv").exists():
    PACKAGE_ROOT = CWD
elif (CWD.parent / "config" / "expected_files.tsv").exists():
    PACKAGE_ROOT = CWD.parent
else:
    raise FileNotFoundError(
        "Could not find config/expected_files.tsv. Run this cell from the "
        "package root or its notebooks/ directory."
    )

WORK_DIR = PACKAGE_ROOT / "work"
WORK_DIR.mkdir(parents=True, exist_ok=True)

# These are the authoritative Portal AnalysisSets specified by the user.
PORTAL_ANALYSIS_SETS = {
    "D0": "IGVFDS0463XTUH",
    "D1": "IGVFDS0991DBQZ",
    "D2": "IGVFDS4402ZGWJ",
    "D3": "IGVFDS2307REVJ",
}


def read_tsv(path: Path) -> pd.DataFrame:
    if not path.is_file():
        return pd.DataFrame()
    return pd.read_csv(path, sep="\t", dtype=str, keep_default_na=False)


def clean_bool(value) -> bool:
    return str(value).strip().lower() in {"true", "1", "yes"}


def parse_alias(value: str) -> str:
    text = str(value or "").strip()
    if not text:
        return ""
    try:
        parsed = json.loads(text)
        if isinstance(parsed, list) and parsed:
            return str(parsed[0])
    except Exception:
        pass
    return text


def aggregate_audits(df: pd.DataFrame, object_type: str | None = None) -> pd.DataFrame:
    if df.empty or "accession" not in df.columns:
        return pd.DataFrame(columns=[
            "accession", "audit_count", "blocker_count", "warning_count",
            "audit_levels", "audit_details"
        ])
    x = df.copy()
    if object_type is not None and "object_type" in x.columns:
        x = x[x["object_type"].eq(object_type)].copy()
    if x.empty:
        return pd.DataFrame(columns=[
            "accession", "audit_count", "blocker_count", "warning_count",
            "audit_levels", "audit_details"
        ])
    x["is_blocker"] = x.get("audit_level", "").isin(["ERROR", "NOT_COMPLIANT"])
    x["is_warning"] = x.get("audit_level", "").eq("WARNING")
    x["audit_line"] = x.apply(
        lambda r: " | ".join(
            p for p in [
                str(r.get("audit_level", "")).strip(),
                str(r.get("audit_category", "")).strip(),
                str(r.get("audit_detail", "")).strip(),
                str(r.get("audit_path", "")).strip(),
            ] if p
        ),
        axis=1,
    )
    return (
        x.groupby("accession", as_index=False)
        .agg(
            audit_count=("accession", "size"),
            blocker_count=("is_blocker", "sum"),
            warning_count=("is_warning", "sum"),
            audit_levels=("audit_level", lambda s: "|".join(sorted(set(v for v in s if v)))),
            audit_details=("audit_line", lambda s: " || ".join(v for v in s if v)),
        )
    )


# -----------------------------------------------------------------------------
# 1. Base 52-row inventory.
# -----------------------------------------------------------------------------
expected = read_tsv(PACKAGE_ROOT / "config" / "expected_files.tsv")
if expected.empty:
    raise FileNotFoundError("config/expected_files.tsv is missing or empty")

summary = expected.copy()
summary["authoritative_file_set"] = summary["day"].map(PORTAL_ANALYSIS_SETS)
if summary["authoritative_file_set"].isna().any():
    bad = sorted(summary.loc[summary["authoritative_file_set"].isna(), "day"].unique())
    raise ValueError(f"Unexpected day labels in expected_files.tsv: {bad}")

# -----------------------------------------------------------------------------
# 2. Source paths. Prefer the generated crosswalk; otherwise reconstruct from
#    source_path_overrides.tsv and source_candidate.
# -----------------------------------------------------------------------------
crosswalk = read_tsv(WORK_DIR / "source_to_submission_crosswalk.tsv")
if crosswalk.empty:
    overrides = read_tsv(PACKAGE_ROOT / "config" / "source_path_overrides.tsv")
    if not overrides.empty:
        crosswalk = expected.merge(
            overrides,
            on=["day", "expected_file_role"],
            how="left",
            suffixes=("", "_override"),
        )
    else:
        crosswalk = expected.copy()

    override = crosswalk.get(
        "source_path_override", pd.Series("", index=crosswalk.index)
    ).fillna("").astype(str).str.strip()
    candidate = crosswalk.get(
        "source_candidate", pd.Series("", index=crosswalk.index)
    ).fillna("").astype(str).str.strip()
    crosswalk["source_path"] = override.where(override.ne(""), candidate)
else:
    if "source_path" not in crosswalk.columns:
        crosswalk["source_path"] = ""

crosswalk["source_exists"] = crosswalk["source_path"].map(
    lambda p: bool(p) and Path(p).is_file()
)
source_cols = [
    c for c in ["alias", "source_path", "source_exists", "conversion_action"]
    if c in crosswalk.columns
]
summary = summary.merge(
    crosswalk[source_cols].drop_duplicates("alias"),
    on="alias",
    how="left",
    suffixes=("", "_crosswalk"),
)
summary["source_path"] = summary.get("source_path", "").fillna("")
summary["source_exists"] = summary.get("source_exists", False).fillna(False).map(clean_bool)
summary["planned_output_exists"] = summary["planned_submission_path"].map(
    lambda p: bool(p) and Path(p).is_file()
)

# -----------------------------------------------------------------------------
# 3. Local validation status.
# -----------------------------------------------------------------------------
validation = read_tsv(WORK_DIR / "validation_report.tsv")
if validation.empty:
    summary["validation_status"] = "NOT_RUN"
    summary["validation_detail"] = "work/validation_report.tsv does not exist; run the validation cell first."
    summary["validated_size_bytes"] = ""
    summary["validated_md5sum"] = ""
else:
    v = validation.rename(columns={
        "size_bytes": "validated_size_bytes",
        "md5sum": "validated_md5sum",
    })
    keep = [
        c for c in [
            "alias", "validation_status", "validation_detail",
            "validated_size_bytes", "validated_md5sum"
        ] if c in v.columns
    ]
    summary = summary.merge(v[keep].drop_duplicates("alias"), on="alias", how="left")
    summary["validation_status"] = summary["validation_status"].replace("", "NOT_RUN").fillna("NOT_RUN")
    summary["validation_detail"] = summary.get("validation_detail", "").fillna("")

# -----------------------------------------------------------------------------
# 4. Existing uploaded-file reconciliation.
# -----------------------------------------------------------------------------
recon = read_tsv(WORK_DIR / "existing_uploaded_reconciliation.tsv")
if recon.empty:
    recon = read_tsv(PACKAGE_ROOT / "config" / "existing_uploaded_reconciliation_template.tsv")
if not recon.empty:
    recon = recon.rename(columns={"expected_alias": "alias"})
    keep = [
        c for c in [
            "alias", "existing_file_accession", "existing_portal_url",
            "reconciliation_status", "audit_counts"
        ] if c in recon.columns
    ]
    summary = summary.merge(recon[keep].drop_duplicates("alias"), on="alias", how="left")
for col in ["existing_file_accession", "existing_portal_url", "reconciliation_status", "audit_counts"]:
    if col not in summary.columns:
        summary[col] = ""
    summary[col] = summary[col].fillna("")

# -----------------------------------------------------------------------------
# 5. Portal audit messages for active AnalysisSets and already-uploaded files.
# -----------------------------------------------------------------------------
audits = read_tsv(WORK_DIR / "portal_audits.tsv")
set_audits = aggregate_audits(audits, "AnalysisSet").rename(columns={
    "accession": "authoritative_file_set",
    "audit_count": "analysis_set_audit_count",
    "blocker_count": "analysis_set_blocker_count",
    "warning_count": "analysis_set_warning_count",
    "audit_levels": "analysis_set_audit_levels",
    "audit_details": "analysis_set_audit_details",
})
summary = summary.merge(set_audits, on="authoritative_file_set", how="left")

file_audits = aggregate_audits(audits, None)
if not file_audits.empty and "object_type" in audits.columns:
    # Remove AnalysisSet rows so the existing-file audit is file-specific.
    file_audits = aggregate_audits(audits[audits["object_type"].ne("AnalysisSet")], None)
file_audits = file_audits.rename(columns={
    "accession": "existing_file_accession",
    "audit_count": "existing_file_audit_count",
    "blocker_count": "existing_file_blocker_count",
    "warning_count": "existing_file_warning_count",
    "audit_levels": "existing_file_audit_levels",
    "audit_details": "existing_file_audit_details",
})
summary = summary.merge(file_audits, on="existing_file_accession", how="left")

for col in [
    "analysis_set_audit_count", "analysis_set_blocker_count", "analysis_set_warning_count",
    "existing_file_audit_count", "existing_file_blocker_count", "existing_file_warning_count",
]:
    if col not in summary.columns:
        summary[col] = 0
    summary[col] = pd.to_numeric(summary[col], errors="coerce").fillna(0).astype(int)
for col in [
    "analysis_set_audit_levels", "analysis_set_audit_details",
    "existing_file_audit_levels", "existing_file_audit_details",
]:
    if col not in summary.columns:
        summary[col] = ""
    summary[col] = summary[col].fillna("")

# -----------------------------------------------------------------------------
# 6. Determine whether READY rows were actually written into upload metadata,
#    and whether those metadata use the correct Portal AnalysisSet IDs.
# -----------------------------------------------------------------------------
eligible = read_tsv(WORK_DIR / "metadata" / "eligible_upload_manifest.tsv")
if not eligible.empty:
    if "alias" not in eligible.columns and "aliases" in eligible.columns:
        eligible["alias"] = eligible["aliases"].map(parse_alias)
    metadata_keep = [c for c in ["alias", "file_set", "submitted_file_name", "md5sum"] if c in eligible.columns]
    eligible = eligible[metadata_keep].rename(columns={
        "file_set": "metadata_file_set",
        "submitted_file_name": "metadata_submitted_file_name",
        "md5sum": "metadata_md5sum",
    })
    eligible["metadata_row_present"] = True
    summary = summary.merge(eligible.drop_duplicates("alias"), on="alias", how="left")
else:
    summary["metadata_row_present"] = False
    summary["metadata_file_set"] = ""
    summary["metadata_submitted_file_name"] = ""
    summary["metadata_md5sum"] = ""

summary["metadata_row_present"] = summary["metadata_row_present"].fillna(False).map(clean_bool)
for col in ["metadata_file_set", "metadata_submitted_file_name", "metadata_md5sum"]:
    if col not in summary.columns:
        summary[col] = ""
    summary[col] = summary[col].fillna("")
summary["metadata_file_set_matches_portal"] = (
    summary["metadata_row_present"]
    & summary["metadata_file_set"].eq(summary["authoritative_file_set"])
)

# -----------------------------------------------------------------------------
# 7. Final upload-readiness classification.
# -----------------------------------------------------------------------------
def classify(row: pd.Series) -> tuple[str, bool, str, str]:
    validation_status = str(row.get("validation_status", "NOT_RUN"))
    upload_action = str(row.get("upload_action", ""))
    recon_status = str(row.get("reconciliation_status", ""))

    if upload_action == "SKIP_ALREADY_UPLOADED_RECONCILE_PORTAL":
        if recon_status == "FOUND_EXACT":
            return (
                "ALREADY_UPLOADED_NO_ACTION",
                False,
                "No upload; review Portal audits and keep the existing accession.",
                "",
            )
        if recon_status in {"NOT_FOUND", "AMBIGUOUS_MULTIPLE"}:
            return (
                "EXISTING_UPLOAD_RECONCILIATION_ISSUE",
                False,
                "Resolve the existing Portal-file match before any upload.",
                f"reconciliation_status={recon_status}",
            )
        return (
            "EXISTING_UPLOAD_RECONCILIATION_PENDING",
            False,
            "Run/review the read-only Portal reconciliation.",
            f"reconciliation_status={recon_status or 'not available'}",
        )

    if upload_action == "BLOCK_UNTIL_DACC_DECISION" or validation_status == "BLOCKED_DECISION":
        return (
            "BLOCKED_DACC_DECISION",
            False,
            "Resolve the DACC representation/content-type decision.",
            str(row.get("validation_detail", "")),
        )

    if validation_status == "INVALID":
        return (
            "INVALID_OUTPUT",
            False,
            "Fix the local validation errors and rerun validation.",
            str(row.get("validation_detail", "")),
        )

    if validation_status == "READY":
        if int(row.get("analysis_set_blocker_count", 0)) > 0:
            return (
                "BLOCKED_ANALYSIS_SET_AUDIT",
                False,
                "Resolve ERROR/NOT_COMPLIANT audits on the target AnalysisSet.",
                str(row.get("analysis_set_audit_details", "")),
            )
        if not bool(row.get("metadata_row_present", False)):
            return (
                "READY_LOCAL_METADATA_NOT_BUILT",
                False,
                "Rebuild draft metadata TSVs after using the Portal AnalysisSet mapping.",
                "Local file passed validation but is absent from eligible_upload_manifest.tsv.",
            )
        if not bool(row.get("metadata_file_set_matches_portal", False)):
            return (
                "BLOCKED_WRONG_ANALYSIS_SET",
                False,
                "Rebuild metadata using the authoritative Portal AnalysisSet for this day.",
                f"metadata_file_set={row.get('metadata_file_set', '')}; expected={row.get('authoritative_file_set', '')}",
            )
        return (
            "READY_TO_UPLOAD",
            True,
            "Include in the next dry-run submission.",
            "",
        )

    # Missing/not-run output: distinguish absent source from unperformed staging.
    source_path = str(row.get("source_path", "")).strip()
    source_exists = bool(row.get("source_exists", False))
    output_exists = bool(row.get("planned_output_exists", False))

    if output_exists and validation_status in {"MISSING", "NOT_RUN", ""}:
        return (
            "VALIDATION_NOT_RUN_OR_STALE",
            False,
            "Rerun the validation cell for the existing staged output.",
            str(row.get("validation_detail", "")),
        )
    if not source_path:
        return (
            "MISSING_SOURCE_PATH",
            False,
            "Add the exact Sherlock path to config/source_path_overrides.tsv.",
            "No source path is recorded.",
        )
    if not source_exists:
        return (
            "MISSING_SOURCE_FILE",
            False,
            "Correct the source path or restore the source file.",
            f"Source not found: {source_path}",
        )
    return (
        "NEEDS_STAGING_OR_CONVERSION",
        False,
        "Enable/re-run staging, then rerun validation.",
        f"Source exists but staged output is not READY; conversion_action={row.get('conversion_action', '')}",
    )

classified = summary.apply(classify, axis=1, result_type="expand")
classified.columns = ["submission_status", "ready_to_upload", "recommended_action", "readiness_issue"]
summary = pd.concat([summary, classified], axis=1)

# Combine all issue text into one searchable field.
def combine_issues(row: pd.Series) -> str:
    parts = []
    for col in [
        "readiness_issue", "validation_detail", "analysis_set_audit_details",
        "existing_file_audit_details"
    ]:
        value = str(row.get(col, "")).strip()
        if value and value not in parts:
            parts.append(value)
    return " || ".join(parts)

summary["all_issues"] = summary.apply(combine_issues, axis=1)
summary = summary.sort_values(["day", "index"], kind="stable")

# -----------------------------------------------------------------------------
# 8. Write consolidated outputs. These are plain files and do not depend on a
#    VS Code/Jupyter rich-output renderer.
# -----------------------------------------------------------------------------
all_path = WORK_DIR / "upload_readiness_all.tsv"
csv_path = WORK_DIR / "upload_readiness_all.csv"
ready_path = WORK_DIR / "upload_ready_only.tsv"
issues_path = WORK_DIR / "upload_missing_and_issues.tsv"
counts_path = WORK_DIR / "upload_readiness_counts.tsv"
md_path = WORK_DIR / "upload_readiness_summary.md"

preferred_columns = [
    "index", "day", "authoritative_file_set", "profile", "expected_file_role",
    "content_type", "alias", "source_path", "source_exists",
    "planned_submission_path", "planned_output_exists", "conversion_action",
    "upload_action", "validation_status", "validation_detail",
    "metadata_row_present", "metadata_file_set", "metadata_file_set_matches_portal",
    "existing_file_accession", "existing_portal_url", "reconciliation_status",
    "analysis_set_blocker_count", "analysis_set_warning_count",
    "existing_file_blocker_count", "existing_file_warning_count",
    "validated_size_bytes", "validated_md5sum", "submission_status",
    "ready_to_upload", "recommended_action", "all_issues",
]
preferred_columns = [c for c in preferred_columns if c in summary.columns]
summary_out = summary[preferred_columns].copy()
summary_out.to_csv(all_path, sep="\t", index=False, lineterminator="\n")
summary_out.to_csv(csv_path, index=False, lineterminator="\n")
summary_out[summary_out["ready_to_upload"]].to_csv(
    ready_path, sep="\t", index=False, lineterminator="\n"
)
summary_out[
    ~summary_out["submission_status"].isin(["READY_TO_UPLOAD", "ALREADY_UPLOADED_NO_ACTION"])
].to_csv(issues_path, sep="\t", index=False, lineterminator="\n")

counts_overall = (
    summary_out.groupby("submission_status", dropna=False)
    .size().rename("count").reset_index()
)
counts_overall.insert(0, "day", "ALL")
counts_by_day = (
    summary_out.groupby(["day", "submission_status"], dropna=False)
    .size().rename("count").reset_index()
)
counts = pd.concat([counts_overall, counts_by_day], ignore_index=True)
counts.to_csv(counts_path, sep="\t", index=False, lineterminator="\n")

now = datetime.now(timezone.utc).isoformat()
ready_n = int(summary_out["ready_to_upload"].sum())
already_n = int(summary_out["submission_status"].eq("ALREADY_UPLOADED_NO_ACTION").sum())
issue_n = int(len(summary_out) - ready_n - already_n)

lines = [
    "# IGVF analysis-file upload readiness summary",
    "",
    f"Generated: `{now}`",
    f"Package root: `{PACKAGE_ROOT}`",
    "",
    "## Authoritative Portal AnalysisSets",
    "",
]
for day, accession in PORTAL_ANALYSIS_SETS.items():
    lines.append(f"- {day}: `{accession}`")
lines.extend([
    "",
    "## Overall counts",
    "",
    f"- Planned rows: **{len(summary_out)}**",
    f"- Ready to upload: **{ready_n}**",
    f"- Already uploaded; no new upload: **{already_n}**",
    f"- Missing, blocked, invalid, or otherwise unresolved: **{issue_n}**",
    "",
    "## Status counts",
    "",
])
for _, r in counts_overall.sort_values("submission_status").iterrows():
    lines.append(f"- {r['submission_status']}: **{int(r['count'])}**")
lines.extend([
    "",
    "## Output files",
    "",
    f"- Full row-level summary: `{all_path}`",
    f"- CSV copy: `{csv_path}`",
    f"- Ready-only manifest: `{ready_path}`",
    f"- Missing/issues-only table: `{issues_path}`",
    f"- Counts by status and day: `{counts_path}`",
    "",
    "A file is classified as `READY_TO_UPLOAD` only when the local file passes validation, "
    "appears in the eligible metadata manifest, uses the authoritative Portal AnalysisSet "
    "for its day, and the target AnalysisSet has no ERROR/NOT_COMPLIANT audit.",
])
md_path.write_text("\n".join(lines) + "\n", encoding="utf-8")

# Plain-text terminal/notebook output; no rich renderer is required.
print("IGVF upload-readiness summary")
print("=" * 31)
print(f"Planned rows:                         {len(summary_out)}")
print(f"Ready to upload:                      {ready_n}")
print(f"Already uploaded; no new upload:      {already_n}")
print(f"Missing/blocked/invalid/unresolved:    {issue_n}")
print()
print(counts_overall.sort_values("submission_status").to_string(index=False))
print()
print("Wrote:")
for p in [all_path, csv_path, ready_path, issues_path, counts_path, md_path]:
    print(f"  {p}")


IGVF upload-readiness summary
Planned rows:                         52
Ready to upload:                      0
Already uploaded; no new upload:      0
Missing/blocked/invalid/unresolved:    52

day                      submission_status  count
ALL                  BLOCKED_DACC_DECISION      4
ALL EXISTING_UPLOAD_RECONCILIATION_PENDING     20
ALL                    MISSING_SOURCE_PATH     16
ALL            NEEDS_STAGING_OR_CONVERSION     12

Wrote:
  /oak/stanford/groups/engreitz/Users/tri/CodeBase/programDE/notebooks/cc_perturbseq_igvf_analysis_prep_4_timepoints/work/upload_readiness_all.tsv
  /oak/stanford/groups/engreitz/Users/tri/CodeBase/programDE/notebooks/cc_perturbseq_igvf_analysis_prep_4_timepoints/work/upload_readiness_all.csv
  /oak/stanford/groups/engreitz/Users/tri/CodeBase/programDE/notebooks/cc_perturbseq_igvf_analysis_prep_4_timepoints/work/upload_ready_only.tsv
  /oak/stanford/groups/engreitz/Users/tri/CodeBase/programDE/notebooks/cc_perturbseq_igvf_analysis_prep_4_time